# Lektion 12 - Reduzierung des Chatverlaufs mit Agent Scratchpad

Dieses Notebook zeigt, wie man den Kontext in langen Unterhaltungen mit dem Microsoft Agent Framework verwaltet. Mit zunehmender Gesprächsdauer steigt die Anzahl der Tokens — was letztlich das Kontextfenster des Modells überschreitet. Wir lösen dies mit einem **Kontext-Zusammenfassungsmuster** und einem **Agent Scratchpad** für persistentes Gedächtnis.

## Was Sie lernen werden:
1. **Warum Kontextverwaltung wichtig ist**: Verständnis von Token-Limits und Kontextfenstern
2. **Kontextbewusste Agenten**: Erstellen von Agenten, die ihren eigenen Gesprächskontext verwalten
3. **Kontext-Zusammenfassungsmuster**: Verwendung von Tools zur Verdichtung des Gesprächsverlaufs
4. **Agent Scratchpad**: Persistentes Gedächtnis, das Kontextreduktion übersteht

## Voraussetzungen:
- Azure OpenAI Einrichtung mit konfigurierten Umgebungsvariablen
- Grundverständnis von Agentenkonzepten aus vorherigen Lektionen


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Warum Kontextmanagement wichtig ist

Jedes LLM hat ein endliches **Kontextfenster** – die maximale Anzahl an Tokens, die es in einer einzelnen Anfrage verarbeiten kann. Im Verlauf eines mehrstufigen Gesprächs:

- **Wächst die Anzahl der Tokens linear** mit jeder Benutzer-Nachricht und Assistenten-Antwort.
- **Dominieren die Prompt-Tokens die Kosten**, weil die gesamte Historie bei jedem Durchgang erneut gesendet wird.
- Überschreitet das Gespräch schließlich **das Kontextfenster**, wird das Modell entweder abschneiden oder einen Fehler ausgeben.

### Strategien zur Verwaltung des Kontexts

| Strategie | Funktionsweise | Kompromiss |
|---|---|---|
| **Abschneiden** | Älteste Nachrichten verwerfen | Frühere Kontexte gehen verloren |
| **Zusammenfassung** | Ältere Nachrichten in einer Zusammenfassung kondensieren | Einige Details gehen verloren, aber wichtige Punkte bleiben erhalten |
| **Notizblock / Externer Speicher** | Wichtige Fakten außerhalb des Gesprächs speichern | Erfordert Toolaufrufe, übersteht aber jede Reduktion |

In diesem Notebook kombinieren wir **Zusammenfassung** mit einem **Notizblock-Tool**, damit der Agent Kontinuität bewahren kann, selbst wenn die Gesprächshistorie verdichtet wird.


## Erstellung eines kontextsensitiven Agenten


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulation eines langen Gesprächs

Lassen Sie uns ein mehrstufiges Gespräch durchgehen, um zu sehen, wie sich der Kontext aufbaut. Der Agent sollte wichtige Details (Vorlieben, Budget, Reisedaten) über die Gesprächsrunden hinweg behalten und Kontinuität zeigen.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Beachten Sie, wie der Agent den Kontext aus früheren Runden beibehält — er weiß über Japan, Sushi, Tempel, Fotografie, das Budget von 3000 $, Alleinreisen und den Zeitraum im April Bescheid. In einem kurzen Gespräch funktioniert das gut, aber wenn das Gespräch länger wird, wird es teuer, die gesamte Historie erneut zu senden.

Lassen Sie uns das Gespräch mit weiteren Runden fortsetzen, um die Kontextansammlung zu beobachten:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontext-Zusammenfassungsmuster

Mit dem Wachstum der Unterhaltung können wir ein **Zusammenfassungstool** verwenden, um den angesammelten Kontext in einem kompakten Format zu verdichten. Der Agent ruft dieses Tool auf, um wichtige Präferenzen zu speichern, sodass selbst wenn ältere Nachrichten entfernt werden, die wesentlichen Informationen erhalten bleiben.

Dieses Muster ist der Baustein für eine ausgefeiltere Reduzierung der Historie:
1. Der Agent identifiziert Schlüsselfakten aus der Unterhaltung
2. Er ruft das Zusammenfassungstool auf, um sie zu speichern
3. Ältere Nachrichten können sicher entfernt werden, weil die Zusammenfassung das Wesentliche festhält

Im Folgenden definieren wir ein `summarize_preferences`-Tool, das der Agent aufrufen kann, um eine kompakte Zusammenfassung dessen zu speichern, was er gelernt hat.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Zusammenfassung

In dieser Lektion haben Sie gelernt, wie man den Kontext in lang andauernden Agenten-Konversationen mit dem Microsoft Agent Framework verwaltet:

### Schlüsselkriterien
- **Kontextfenster sind begrenzt** — jeder Token in der Gesprächshistorie kostet Geld und zählt zum Limit.
- **Zusammenfassungstools** ermöglichen es dem Agenten, den angesammelten Kontext in kompakte Zusammenfassungen zu verdichten, wodurch der Tokenverbrauch reduziert und wesentliche Informationen bewahrt werden.
- **Agent-Scratchpads** bieten einen persistenten externen Speicher, der jede Gesprächsreduktion übersteht.

### Was Sie gebaut haben
- Einen **kontextbewussten Agenten**, der Kontinuität über mehrstufige Unterhaltungen hinweg aufrechterhält
- Ein **Zusammenfassungstool** (`summarize_preferences`), das wichtige Benutzerdetails in einem kompakten Format speichert
- Eine **mehrstufige Konversation**, die Kontextbeibehaltung und Umgang mit Änderungen demonstriert

### Anwendungsbeispiele aus der Praxis
- **Kundendienst-Bots**: Erinnern sich an Präferenzen über lange Support-Sitzungen hinweg
- **Persönliche Assistenten**: Verfolgen laufende Projekte, ohne den Kontext neu erklären zu müssen
- **Bildungs-Tutoren**: Halten den Lernfortschritt der Schüler über viele Interaktionen hinweg fest

### Nächste Schritte
- Implementieren eines vollständigen Scratchpad-Tools mit dateibasierter Persistenz
- Automatische Verkürzung der Historie nach der Zusammenfassung hinzufügen
- Kombination mit Vektordatenbanken für die semantische Speicher-Suche
- Agenten bauen, die Konversationen Tage später mit vollem Kontext fortsetzen können


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mithilfe des KI-Übersetzungsdienstes [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ausgangssprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung resultieren.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
